In [1]:
%pip install \
    huggingface_hub \
    wandb \
    langchain-community \
    langgraph \
    geopy \
    langchain_google_genai \
    langchain-openai \
    NRCLex \
    --upgrade \
    --quiet \
    --no-cache-dir

%pip install \
    llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 \
    --quiet \
    --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 16.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 254.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 316.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.4/245.4 kB 358.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 166.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 295.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 274.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 287.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 388.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.2/552.2 kB 377.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 259.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 208.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [2]:
import numpy as np 
import pandas as pd 
import wandb
from tqdm.auto import tqdm

In [3]:
from pathlib import Path
import gc
import json
import os
import re
import time
import sys
from langchain_core.callbacks import UsageMetadataCallbackHandler

from typing import Any, Literal, TypedDict

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

os.environ["HF_TOKEN"] = user_secrets.get_secret("hf_hub")
os.environ["OPENAI_API_KEY"] = user_secrets.get_secret("openai")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("wandb")

In [4]:
%cd /kaggle/working

!rm -rf iad2026-mapping-Russian-Songs
!git clone https://github.com/kholodovTimur/iad2026-mapping-Russian-Songs.git

PROJECT_ROOT = Path("/kaggle/working/iad2026-mapping-Russian-Songs")


sys.path.append("/kaggle/working/iad2026-mapping-Russian-Songs")
from CoderAgent import SongToponymGeoRecognition


/kaggle/working
Cloning into 'iad2026-mapping-Russian-Songs'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 65 (delta 30), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 424.74 KiB | 3.34 MiB/s, done.
Resolving deltas: 100% (30/30), done.


Cloning into 'iad2026-mapping-Russian-Songs'...


In [14]:
def make_expirement(self, data: pd.DataFrame, run_name: str, project_name: str = "iadProject", notes: str = ""):
    
    def parse_toponyms(cell) -> list[dict]:
            if pd.isna(cell):
                return []
    
            return [
                {
                    "toponym": item.split(":")[0],
                    "type": item.split(":")[-1],
                }
                for item in cell.split(", ")
            ]

    def calculate_metrics(df: pd.DataFrame) -> dict:
        accuracy = len(
            df[
                (df["FN"] == 0)
                & (df["FP"] == 0)
            ]
        ) / len(df)

        tp_sum = df["TP"].sum(skipna=True)
        fn_sum = df["FN"].sum(skipna=True)
        fp_sum = df["FP"].sum(skipna=True)

        recall = tp_sum / (tp_sum + fn_sum) if tp_sum + fn_sum > 0 else 0
        precision = tp_sum / (tp_sum + fp_sum) if tp_sum + fp_sum > 0 else 0

        f_score = (
            2 * precision * recall / (precision + recall)
            if precision and recall
            else 0
        )

        return {
            "accuracy": accuracy,
            "recall": recall,
            "precision": precision,
            "f": f_score,
        }

    
    
    df = data.copy()

    df["Топонимы"] = df["Топонимы"].apply(parse_toponyms)

    result_columns = [
        "tonality",
        "predicted",
        "true_toponyms",
        "true_types",
        "predicted_toponyms",
        "predicted_types",
        "geo_objects",
        "geo_latitudes",
        "geo_longitudes",
        "geo_geometries",
        "geo_addresses",
        "true_predicted",
        "unpredicted",
        "overpredicted",
        "TP",
        "FP",
        "FN",
    ]

    for column in result_columns:
        df[column] = None
        df[column] = df[column].astype(object)

    hyperparameters = {
        "model": run_name or self.recognition_model_name,
        "temp_recog": self.recognition_t,
        "temp_geo": self.geo_t,
        "prompt": "-",
        "context_window_recog": self.recog_ctx,
        "context_window_geo": self.geo_ctx,
        "max_tokens_recog": self.recognition_mt,
        "max_tokens_geo": self.geo_mt,
        "local": str(self.local)
    }

    callback = UsageMetadataCallbackHandler()

    with wandb.init(
        project=project_name,
        config=hyperparameters,
        name=run_name or self.recognition_model_name,
        notes=notes,
    ) as run:
        run.define_metric("technical/total_time", summary="mean")
        run.define_metric("technical/recognition_time", summary="mean")
        run.define_metric("technical/geo_time", summary="mean")

        errors_count = 0
        
        with tqdm(df.iterrows(), total=len(df)) as pbar:
            for index, row in pbar:
                start_time = time.time()

                try:
                    if self.local == True:
                        pred_state = self.model.invoke(
                            {
                                "song_text": row["lyrics"],
                            }
                        )
                    else: 
                        pred_state = self.model.invoke(
                            {
                                "song_text": row["lyrics"]
                            }, config={"callbacks": [callback]}
                        )

                    if len(pred_state["toponymns"].places) == 0:
                        predicted_toponyms = []
                        geo_objects = []
                        tonality_info = {}
                        recognition_time = pred_state.get("recognitiontime", np.nan)
                        geo_time = np.nan
                    else:
                        predicted_toponyms = pred_state["confident_topomymns"]
                        geo_objects = pred_state.get("geo_objects", [])
                        tonality_info = pred_state.get("tonality_dict", {})
                        recognition_time = pred_state["recognitiontime"]
                        geo_time = pred_state["geotime"]

                    true_toponyms = [
                        item["toponym"]
                        for item in row["Топонимы"]
                    ]

                    df.at[index, "predicted"] = predicted_toponyms
                    df.at[index, "true_toponyms"] = true_toponyms
                    df.at[index, "predicted_toponyms"] = predicted_toponyms
                    df.at[index, "geo_objects"] = geo_objects
                    df.at[index, "geo_latitudes"] = [item.get("latitude") for item in geo_objects]
                    df.at[index, "geo_longitudes"] = [item.get("longitude") for item in geo_objects]
                    df.at[index, "geo_geometries"] = [item.get("geometry") for item in geo_objects]
                    df.at[index, "geo_addresses"] = [item.get("address") for item in geo_objects]
                    df.at[index, "tonality"] = tonality_info

                    df.at[index, "true_predicted"] = (
                        set(true_toponyms) & set(predicted_toponyms)
                    )
                    df.at[index, "unpredicted"] = (
                        set(true_toponyms) - set(predicted_toponyms)
                    )
                    df.at[index, "overpredicted"] = (
                        set(predicted_toponyms) - set(true_toponyms)
                    )

                    df.at[index, "TP"] = len(df.at[index, "true_predicted"])
                    df.at[index, "FN"] = len(df.at[index, "unpredicted"])
                    df.at[index, "FP"] = len(df.at[index, "overpredicted"])

                    total_time = time.time() - start_time

                except Exception as error:
                    total_time = time.time() - start_time
                    recognition_time = np.nan
                    geo_time = np.nan

                    for column in result_columns:
                        df.at[index, column] = np.nan

                    errors_count += 1

                    tqdm.write(f"Error: {error}")

                pbar.set_postfix(
                    {
                        "TP": df.at[index, "TP"],
                        "FN": df.at[index, "FN"],
                        "FP": df.at[index, "FP"],
                    }
                )

                metrics = calculate_metrics(df.loc[:index])

                if self.local == False:
                    usage = {k:i for v in callback.usage_metadata.values() for k, i in v.items()} 
                    try:
                        cache = usage['input_token_details']['cache_read']
                    except:
                        cache = 0
                    input_tokens = usage['input_tokens'] - cache
                    output_tokens = usage['output_tokens']
                else:
                    cache = 0 
                    input_tokens = 0
                    output_tokens = 0

                run.log(
                    {
                        "text/accuracy": metrics["accuracy"],
                        "toponyms/recall": metrics["recall"],
                        "toponyms/precision": metrics["precision"],
                        "toponyms/f": metrics["f"],
                        "technical/error_rate": errors_count,
                        "technical/total_time": total_time,
                        "technical/recognition_time": recognition_time,
                        "technical/geo_time": geo_time,
                        "usage/cache": cache,
                        "usage/input_tokens": input_tokens,
                        "usage/output_tokens": output_tokens
                        
                    }
                )
                    
        artifact = wandb.Artifact("words", type="dataset")
        words = wandb.Table(
            dataframe=df[
                [
                    "unpredicted",
                    "overpredicted",
                ]
            ]
        )

        artifact.add(words, "unpredicted_overpredicted")
        wandb.log_artifact(artifact)


    return df

In [10]:
DATA_PATH = PROJECT_ROOT / "data" / "test_data.csv"

dataset = pd.read_csv(DATA_PATH)

dataset.shape

(312, 7)

---

In [94]:
model = SongToponymGeoRecognition(local = False,
                                  recognition_model_name = "gpt-5.4-mini",
                                  geo_model_name  = "gpt-5.4-mini",
                                  recognition_mt = 512,
                                  recognition_t = 0.7,
                                  recog_ctx = 4096,
                                  geo_mt = 512,
                                  geo_t = 0.1,
                                  geo_ctx = 4096,
                                  verbose = False)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: blanchefort/rubert-base-cased-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2-cedr-emotion-detection
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: Aniemore/rubert-tiny2-russian-emotion-detection
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


---

In [95]:
new_df = make_expirement(model,
    data=dataset.copy(),
    run_name="gpt-5.4-mini",
    notes="Final experiment"
)

  0%|          | 0/312 [00:00<?, ?it/s]

technical/error_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
technical/geo_time,▁▄▂▃▇▆▃▁▂▆▃▂▂▄█▁▂▄▁▃▃▂▂▁▂ ▁ ▁
technical/recognition_time,▃▃▂▁▄▃▄▃▂▄▂▂▂▃▃▅▄▂▁▃▅▁▂▂█▂▂▂▂▃▂▁▁▁▁▁▁▁▂▁
technical/total_time,▂▃▃▂▂▅▄▄▇▂▃▂▄▂█▂▃▆▃▃▁▂▂▂▄▂▁▄▁▁▁▂▁▁▁▁▁▂▁▁
text/accuracy,█▆▅▄▃▃▃▂▁▁▁▁▁▁▁▁▂▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄
toponyms/f,▇▇██▃▁▃▃▂▂▂▃▃▄▄▃▃▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
toponyms/precision,█▆▅▁▁▂▁▁▁▁▂▂▂▂▂▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
toponyms/recall,█▆▄▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
usage/cache,▁▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇█████
usage/input_tokens,▁▁▁▂▂▂▂▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇███████
+1,...


In [96]:
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUTPUT_DIR / "qgt_5_4_mini_result_full_with_coords_geometry.csv"

def serialize_value(value):
    if isinstance(value, set):
        value = sorted(value)

    if isinstance(value, tuple):
        value = list(value)

    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False, default=str)

    if pd.isna(value):
        return ""

    return value

csv_df = new_df.copy()
for column in csv_df.columns:
    csv_df[column] = csv_df[column].apply(serialize_value)

csv_df.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(f"Готово: {len(csv_df)} строк")
print(f"CSV сохранён сюда: {CSV_PATH}")

Готово: 312 строк
CSV сохранён сюда: /kaggle/working/outputs/qgt_5_4_mini_result_full_with_coords_geometry.csv


---

In [ ]:
model_2 = SongToponymGeoRecognition(local = True,
                                  recognition_model_repo_id = "unsloth/Qwen3.6-35B-A3B-GGUF",
                                  recognition_model_name = "Qwen3.6-35B-A3B-UD-Q5_K_M.gguf",
                                  geo_model_repo_id = "unsloth/Qwen3.6-35B-A3B-GGUF",
                                  geo_model_name = "Qwen3.6-35B-A3B-UD-Q5_K_M.gguf",
                                  recognition_mt = 512,
                                  recognition_t = 0.7,
                                  recog_ctx = 4096,
                                  geo_mt = 512,
                                  geo_t = 0.1,
                                  geo_ctx = 4096,
                                  verbose = False)

In [17]:
new_df_qwen = make_expirement(model_2,
    data=dataset.copy(),
    run_name="Qwen3.6-35B-A3B-GGUF",
    notes="Final experiment"
)

  0%|          | 0/312 [00:00<?, ?it/s]

Error: 1 validation error for SongInfo
  Invalid JSON: EOF while parsing a string at line 1 column 1765 [type=json_invalid, input_value='{"places": [{"toponym": ...ад Петра", "type', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Error: Requested tokens (5117) exceed context window of 4096
Error: 1 validation error for SongInfo
  Invalid JSON: EOF while parsing a string at line 1 column 1556 [type=json_invalid, input_value='{ "places": [ { "toponym...бокоп", "type": "', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Error: 1 validation error for SongInfo
  Invalid JSON: EOF while parsing an object at line 1 column 1774 [type=json_invalid, input_value='{ "places": [ { "toponym... "type": "другое"', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Error: 1 validation error for SongInfo
  Invalid JSON: EOF while parsing a value at line 1 co

RateLimiter caught an error, retrying (0/2 tries). Called with (*('улица Суворова, 13',), **{'geometry': 'geojson', 'exactly_one': True}).
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 1450, in getresponse
    response.begin()
  File "/usr/lib/python3.12/http/client.py", line 336, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 297, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket

Error: 1 validation error for SongInfo
  Invalid JSON: EOF while parsing a string at line 1 column 1650 [type=json_invalid, input_value='{ "places": [ { "toponym...яткино", "normal', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Error: Requested tokens (4734) exceed context window of 4096
Error: 1 validation error for SongInfo
  Invalid JSON: EOF while parsing a value at line 1 column 1745 [type=json_invalid, input_value='{ "places": [ { "toponym..."Крылатском",', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Error: 1 validation error for SongInfo
  Invalid JSON: EOF while parsing a string at line 1 column 1683 [type=json_invalid, input_value='{"places": [{"toponym": ...: "Мурманск", "', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid


technical/error_rate,▁▁▂▂▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇▇████████████
technical/geo_time,▁▁▁ ▁▃▃ ▁▁█▁▃ ▁▂ ▅▂▂▄▂▄▁▂▂▂▁ ▂ ▁
technical/recognition_time,▃▃▅▃ ▂▁▆▃▁█ ▇█▃▁▂█▃▆▂▂▁▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
technical/total_time,▃▃█▂▂▁█▅▅▅▁▇▁▃▄▂▆▁▂▃▃▂▃▁▂▂▁▁▁▁▃▁▁▂▁▁▁▁▁▁
text/accuracy,▃▁▂▁▃▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▅▆▆▇▇▇████
toponyms/f,▁▄▃▆▄▆█▇▂▆▆▅▅▄▄▄▅▅▅▆▆▆▆▆▇██▇████████████
toponyms/precision,▃▂▃▂▂▁▂▂▃▅▅▅▆▆▆▇▇▇▇▇▇▇▇███████▇▇▇▇▇▇▇▇▇▇
toponyms/recall,▁▅▆▇▆███▆▆▆▆▆▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
usage/cache,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
usage/input_tokens,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+1,...
